# Preparación, limpieza y estructuración de datos
## Proyecto: Emisiones de CO₂ de vehículos

**Evidencia:** GA1-220501094-01-AA1-EV07

**Pregunta analítica:** ¿Qué características del vehículo (cilindrada, cilindros, tipo de combustible, transmisión y clase) influyen más en sus emisiones de CO₂ por kilómetro?

**Datasets (en `data/raw/`, sin modificar):**

| Dataset | Archivo | Origen | Unidad de CO₂ |
| --- | --- | --- | --- |
| `co2_raw` | `CO2 Emissions_Canada.csv` | Kaggle, datos del Gobierno de Canadá (modelos 2014–2020) | g/km |
| `epa_raw` | `fuel_economy_epa.csv` | Data.gov / EPA, fueleconomy.gov (modelos 1984–2027) | g/milla |

**Contenido del notebook:**
1. Configuración y carga
2. Lección 2: limpieza y estandarización (pipeline con `pipe`)
3. Lección 3: filtrado, ordenamiento, agrupación e indicadores
4. Lección 4: combinación, validación y exportación
5. Comparación de la calidad antes y después

## 1. Configuración y carga de datos

In [39]:
from pathlib import Path

import numpy as np
import pandas as pd

# Rutas relativas a la carpeta notebooks/
RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(exist_ok=True)   # crea la carpeta si no existe

pd.set_option('display.max_columns', 30)
print('pandas', pd.__version__)

pandas 3.1.0


In [40]:
# Datos originales: no se modifican, se trabaja sobre copias
co2_raw = pd.read_csv(RAW / 'CO2 Emissions_Canada.csv')
epa_raw = pd.read_csv(RAW / 'fuel_economy_epa.csv', low_memory=False)

print('co2_raw:', co2_raw.shape)
print('epa_raw:', epa_raw.shape)

co2_raw: (7385, 12)
epa_raw: (50407, 84)


### 1.1 Estado inicial de la calidad

Antes de limpiar, se toma una "foto" de cada dataset. En la EPA, los duplicados se cuentan **sin la columna `id`**, porque `id` es único para cada fila y oculta los registros repetidos (hallazgo del diagnóstico de calidad).

In [41]:
def quality_snapshot(df, name, ignore=None):
    """Resume filas, columnas, duplicados, nulos y memoria de una tabla."""
    subset = df.columns.drop(ignore) if ignore else None
    return {
        'dataset': name,
        'filas': len(df),
        'columnas': df.shape[1],
        'duplicados': int(df.duplicated(subset=subset).sum()),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    }

quality_before = pd.DataFrame([
    quality_snapshot(co2_raw, 'co2_canada'),
    quality_snapshot(epa_raw, 'epa', ignore=['id']),
])
quality_before

,dataset,filas,columnas,duplicados,nulos,memoria_mb
0,co2_canada,7385,12,1103,0,2.5
1,epa,50407,84,40,543858,82.3


## 2. Lección 2: limpieza y estandarización

Para cada problema hay tres opciones: **eliminar, corregir o conservar**. Primero se **detecta** y se analiza la causa con los datos crudos. Después, cada corrección se implementa como una función y todas se encadenan en un pipeline (sección 2.6).

### 2.1 Duplicados exactos en `co2_raw`

In [42]:
dup_co2 = co2_raw[co2_raw.duplicated(keep=False)].sort_values(['Make', 'Model'])
print('Filas sobrantes:', co2_raw.duplicated().sum())
print('Filas implicadas (originales + copias):', len(dup_co2))
dup_co2[['Make', 'Model', 'Engine Size(L)', 'Transmission', 'CO2 Emissions(g/km)']].head(6)

Filas sobrantes: 1103
Filas implicadas (originales + copias): 2102


,Make,Model,Engine Size(L),Transmission,CO2 Emissions(g/km)
3313,ACURA,ILX,2.4,AM8,192
4371,ACURA,ILX,2.4,AM8,192
5454,ACURA,ILX,2.4,AM8,199
6503,ACURA,ILX,2.4,AM8,199
3316,ACURA,MDX HYBRID AWD,3.0,AM7,210
4374,ACURA,MDX HYBRID AWD,3.0,AM7,210


**Causa:** el dataset de Kaggle junta 7 años de modelo (2014–2020) **sin columna de año**. Cuando un mismo vehículo se vendió igual en varios años, aparece repetido con exactamente los mismos valores y no hay forma de distinguir las filas.

**Decisión: eliminar** las 1 103 copias. La pregunta es sobre las *características* del vehículo, y cada configuración debe contar una sola vez. Si se conservan, las configuraciones que más años duraron en el mercado pesarían más en los promedios.

### 2.2 Duplicados ocultos en `epa_raw`

In [43]:
cols_sin_id = epa_raw.columns.drop('id')
print('id es único:', epa_raw['id'].is_unique)
print('Duplicados sin contar id:', epa_raw.duplicated(subset=cols_sin_id).sum())

dup_epa = epa_raw[epa_raw.duplicated(subset=cols_sin_id, keep=False)].sort_values(['year', 'make', 'model'])
dup_epa[['id', 'year', 'make', 'model', 'displ', 'trany', 'co2TailpipeGpm']].head(6)

id es único: True
Duplicados sin contar id: 40


,id,year,make,model,displ,trany,co2TailpipeGpm
19211,27459,1984,Jeep,J-10 Pickup 4WD,5.9,Automatic 3-spd,807.909091
19213,27460,1984,Jeep,J-10 Pickup 4WD,5.9,Automatic 3-spd,807.909091
7511,1676,1985,Subaru,Sedan 4WD,1.8,Automatic 3-spd,403.954545
7522,1677,1985,Subaru,Sedan 4WD,1.8,Automatic 3-spd,403.954545
20879,2897,1987,CX Automotive,CX 25Tri,2.5,Manual 5-spd,522.764706
20890,2898,1987,CX Automotive,CX 25Tri,2.5,Manual 5-spd,522.764706


**Causa:** son el mismo vehículo registrado dos veces con distinto `id`; las otras 83 columnas son idénticas.

**Decisión: eliminar** las 40 copias y conservar la primera aparición.

### 2.3 Valores faltantes: comprobar la causa antes de decidir

`co2_raw` no tiene nulos. En `epa_raw` se analizan las columnas que usará el proyecto.

In [44]:
cols_interes = ['displ', 'cylinders', 'trany', 'drive', 'fuelType1', 'atvType', 'co2', 'co2TailpipeGpm']
epa_raw[cols_interes].isna().sum()

displ              1653
cylinders          1655
trany                11
drive              1186
fuelType1             2
atvType           43547
co2                   0
co2TailpipeGpm        0
dtype: int64

**Hipótesis 1:** `displ` (cilindrada) y `cylinders` están vacíos porque el vehículo no tiene motor de combustión (eléctricos `EV` y de hidrógeno `FCV`/`eFCV`).

In [45]:
sin_motor = epa_raw['atvType'].isin(['EV', 'FCV', 'eFCV'])
displ_nulo = epa_raw['displ'].isna()

print('Nulos de displ:', displ_nulo.sum())
print('Vehículos sin motor de combustión:', sin_motor.sum())
print('Coinciden fila a fila:', (displ_nulo == sin_motor).all())
epa_raw.loc[displ_nulo & ~sin_motor, ['year', 'make', 'model', 'fuelType', 'displ', 'cylinders']]

Nulos de displ: 1653
Vehículos sin motor de combustión: 1652
Coinciden fila a fila: False


,year,make,model,fuelType,displ,cylinders
21410,1985,Subaru,RX Turbo,Regular,NaN,NaN
21411,1985,Subaru,RX Turbo,Regular,NaN,NaN


La hipótesis se cumple casi completa: 1 651 de los 1 653 nulos son vehículos sin motor de combustión. Las 2 excepciones son el **Subaru RX Turbo de 1985**, un vehículo a gasolina al que no se le registró la cilindrada. Además, un solo eléctrico sí trae cilindrada; por eso la comparación fila a fila da `False`.

**Decisión: conservar los nulos.** Rellenar con 0 inventaría un motor de 0 litros. Los vehículos sin motor de combustión se excluirán más adelante del análisis de emisiones, porque su CO₂ por el escape es 0 y no responden a la pregunta sobre características del motor.

**Hipótesis 2:** la columna `co2` usa `-1` como código de "sin dato" (valor centinela, encontrado en el diagnóstico).

In [46]:
centinela = epa_raw['co2'] == -1
print('Filas con co2 = -1:', centinela.sum())
print(centinela.groupby(epa_raw['year'] >= 2013).mean().rename({False: 'antes de 2013', True: '2013 en adelante'}))
print('Nulos o ceros en co2TailpipeGpm (vehículos con motor):',
      ((epa_raw['co2TailpipeGpm'].isna() | (epa_raw['co2TailpipeGpm'] <= 0)) & ~sin_motor).sum())

Filas con co2 = -1: 31939
year
antes de 2013       0.997751
2013 en adelante    0.000000
Name: co2, dtype: float64
Nulos o ceros en co2TailpipeGpm (vehículos con motor): 0


El `-1` aparece en casi todos los modelos anteriores a 2013 y en ninguno posterior: la EPA empezó a publicar ese campo en 2013. Es un **nulo disfrazado** que bajaría cualquier promedio.

**Decisiones:**
- **Corregir** el `-1` por un nulo real (`pd.NA`) en `co2_g_mi_reported`.
- Usar `co2TailpipeGpm` como medida principal de CO₂: está completa para todos los vehículos con motor. Se convierte de g/milla a g/km (÷ 1,609344) para que sea comparable con Canadá.
- `atvType` vacío significa vehículo **convencional** (sin tecnología alternativa). Se **corrige** con el valor explícito `'Conventional'`.
- Los 11 nulos de `trany` (eléctricos antiguos y dos camionetas de 1984) y los de `drive` (casi todos de 1984) se **conservan**: no se usan como clave y su peso es mínimo.

### 2.4 Estandarización de texto y categorías

In [47]:
# Espacios sobrantes en los modelos
for name, df, col in [('co2', co2_raw, 'Model'), ('epa', epa_raw, 'model')]:
    print(name,
          '| espacios dobles:', df[col].str.contains('  ', regex=False).sum(),
          '| espacios al inicio/fin:', (df[col] != df[col].str.strip()).sum(),
          '| espacio invisible:', df[col].str.contains('\xa0', regex=False).sum())

# Mayúsculas distintas en las marcas: 'ACURA' en Canadá y 'Acura' en la EPA
print(co2_raw['Make'].head(2).tolist(), epa_raw['make'].head(2).tolist())

co2 | espacios dobles: 5 | espacios al inicio/fin: 5 | espacio invisible: 0


epa | espacios dobles: 90 | espacios al inicio/fin: 0 | espacio invisible: 0
['ACURA', 'ACURA'] ['Alfa Romeo', 'Ferrari']


**Decisión: corregir.** Se quitan espacios sobrantes e invisibles, y marca y modelo quedan en mayúsculas en ambos datasets para que se puedan comparar.

#### Duplicados implícitos

Al estandarizar el texto pueden aparecer duplicados que antes no se veían, por ejemplo `Fusion Hybrid` y `FUSION HYBRID`:

In [48]:
co2_sin_dup = co2_raw.drop_duplicates()
modelo_std = co2_sin_dup['Model'].str.replace(r'\s+', ' ', regex=True).str.strip().str.upper()
implicitos = co2_sin_dup.assign(Model=modelo_std)

print('Duplicados implícitos en co2:', implicitos.duplicated().sum())
ejemplo = implicitos.duplicated(keep=False) & (modelo_std == 'FUSION HYBRID')
co2_sin_dup.loc[ejemplo, ['Make', 'Model', 'Transmission', 'Fuel Type', 'CO2 Emissions(g/km)']]

Duplicados implícitos en co2: 294


,Make,Model,Transmission,Fuel Type,CO2 Emissions(g/km)
3702,FORD,FUSION HYBRID,AV,X,131
4779,FORD,FUSION HYBRID,AV,X,131
5798,FORD,Fusion Hybrid,AV,X,131
6816,FORD,Fusion Hybrid,AV,X,131


**Causa:** algunos años del dataset escriben el modelo en mayúsculas y otros no. Son el mismo vehículo en dos años distintos, el mismo caso de los duplicados exactos.

**Decisión: eliminar**, con la misma justificación de la sección 2.1. Por eso el pipeline quita duplicados **otra vez después** de estandarizar el texto.

Las **transmisiones** y las **clases de vehículo** se escriben distinto en cada fuente:

In [49]:
print('Transmisiones Canadá:', co2_raw['Transmission'].nunique(), co2_raw['Transmission'].unique()[:8].tolist())
print('Transmisiones EPA:', epa_raw['trany'].nunique(), epa_raw['trany'].dropna().unique()[:6].tolist())
print('Clases Canadá:', co2_raw['Vehicle Class'].nunique(), '| Clases EPA:', epa_raw['VClass'].nunique())

Transmisiones Canadá: 27 ['AS5', 'M6', 'AV7', 'AS6', 'AM6', 'A6', 'AM7', 'AV8']
Transmisiones EPA: 40 ['Manual 5-spd', 'Automatic 3-spd', 'Automatic 4-spd', 'Automatic 5-spd', 'Manual 4-spd', 'Manual 3-spd']
Clases Canadá: 16 | Clases EPA: 34


En Canadá la transmisión es un código: letra(s) del tipo (A = automática, AS = automática con selector, AM = manual automatizada, AV = variable continua, M = manual) + número de marchas. En la EPA es texto libre como `Automatic 4-spd` o `Automatic (S8)`.

**Decisión: corregir** creando columnas nuevas y conservando la original:
- `transmission_type`: `Automatic`, `Manual` o `CVT`.
- `gears`: número de marchas. Queda nulo cuando la caja es de variación continua sin marchas fijas.
- `vehicle_group`: agrupa las 16 clases de Canadá y las 34 de la EPA en 8 grupos comunes (Car, SUV, Pickup, Minivan, Van, Station wagon, Special purpose, Two-seater).

### 2.5 Funciones de limpieza reutilizables

Cada función recibe un DataFrame y devuelve otro; así se pueden encadenar con `pipe`.

In [50]:
# ---------- Funciones comunes ----------
def remove_duplicates(df, ignore=None):
    """Elimina filas repetidas; 'ignore' excluye columnas de la comparación (ej. id)."""
    subset = df.columns.drop(ignore) if ignore else None
    return df.drop_duplicates(subset=subset).reset_index(drop=True)


def clean_text(df, columns):
    """Quita espacios invisibles y sobrantes y pasa el texto a mayúsculas."""
    df = df.copy()
    for col in columns:
        df[col] = (df[col]
                   .str.replace('\xa0', ' ', regex=False)
                   .str.replace(r'\s+', ' ', regex=True)
                   .str.strip()
                   .str.upper())
    return df


def add_vehicle_group(df, class_col='vehicle_class'):
    """Agrupa las clases de vehículo de ambas fuentes en categorías comunes."""
    df = df.copy()
    cls = df[class_col].str.upper()
    conditions = [
        cls.str.contains('PICKUP'),
        cls.str.contains('SPORT UTILITY|SUV'),
        cls.str.contains('MINIVAN'),
        cls.str.contains('VAN'),
        cls.str.contains('WAGON'),
        cls.str.contains('SPECIAL PURPOSE'),
        cls.str.contains('TWO'),
    ]
    groups = ['Pickup', 'SUV', 'Minivan', 'Van', 'Station wagon', 'Special purpose', 'Two-seater']
    df['vehicle_group'] = np.select(conditions, groups, default='Car')
    return df

In [51]:
# ---------- Funciones para co2_canada ----------
CO2_COLUMNS = {
    'Make': 'make', 'Model': 'model', 'Vehicle Class': 'vehicle_class',
    'Engine Size(L)': 'engine_size_l', 'Cylinders': 'cylinders',
    'Transmission': 'transmission', 'Fuel Type': 'fuel_code',
    'Fuel Consumption City (L/100 km)': 'fuel_city_l100km',
    'Fuel Consumption Hwy (L/100 km)': 'fuel_hwy_l100km',
    'Fuel Consumption Comb (L/100 km)': 'fuel_comb_l100km',
    'Fuel Consumption Comb (mpg)': 'fuel_comb_mpg',
    'CO2 Emissions(g/km)': 'co2_g_km',
}

TRANSMISSION_TYPES = {'A': 'Automatic', 'AS': 'Automatic', 'AM': 'Automatic',
                      'AV': 'CVT', 'M': 'Manual'}


def rename_co2(df):
    """Nombres de columna en inglés, minúsculas y sin espacios ni unidades entre paréntesis."""
    return df.rename(columns=CO2_COLUMNS)


def parse_transmission_co2(df):
    """Separa el código (ej. 'AS6') en tipo de transmisión y número de marchas."""
    df = df.copy()
    parts = df['transmission'].str.extract(r'^([A-Z]+)(\d*)$')
    df['transmission_type'] = parts[0].map(TRANSMISSION_TYPES)
    df['gears'] = pd.to_numeric(parts[1], errors='coerce').astype('Int8')
    return df


def types_co2(df):
    return df.astype({
        'make': 'category', 'vehicle_class': 'category', 'transmission': 'category',
        'fuel_code': 'category', 'transmission_type': 'category', 'vehicle_group': 'category',
        'engine_size_l': 'float32', 'cylinders': 'int8',
        'fuel_city_l100km': 'float32', 'fuel_hwy_l100km': 'float32', 'fuel_comb_l100km': 'float32',
        'fuel_comb_mpg': 'int16', 'co2_g_km': 'int16',
    })

In [52]:
# ---------- Funciones para epa ----------
EPA_COLUMNS = {
    'id': 'id', 'year': 'year', 'make': 'make', 'model': 'model', 'VClass': 'vehicle_class',
    'drive': 'drive', 'displ': 'engine_size_l', 'cylinders': 'cylinders', 'trany': 'transmission',
    'fuelType': 'fuel_type_detail', 'fuelType1': 'fuel_type', 'atvType': 'technology',
    'city08': 'city_mpg', 'highway08': 'hwy_mpg', 'comb08': 'comb_mpg',
    'co2': 'co2_g_mi_reported', 'co2TailpipeGpm': 'co2_g_mi',
}
KM_PER_MILE = 1.609344
NO_ENGINE = ['EV', 'FCV', 'eFCV']


def select_epa(df):
    """Conserva solo las 17 columnas útiles para la pregunta (de 84) y las renombra."""
    return df[list(EPA_COLUMNS)].rename(columns=EPA_COLUMNS)


def fix_epa_values(df):
    """Corrige el centinela -1, rellena technology y convierte unidades."""
    df = df.copy()
    df['co2_g_mi_reported'] = df['co2_g_mi_reported'].mask(df['co2_g_mi_reported'] == -1)
    df['technology'] = df['technology'].fillna('Conventional')
    df['fuel_type'] = df['fuel_type'].str.capitalize()        # 'Regular Gasoline' -> 'Regular gasoline'
    df['co2_g_km'] = (df['co2_g_mi'] / KM_PER_MILE).round(1)
    # L/100 km = 235.215 / mpg (galón estadounidense). No aplica a vehículos sin motor (usan MPGe)
    con_motor = ~df['technology'].isin(NO_ENGINE)
    df['fuel_comb_l100km'] = (235.215 / df['comb_mpg']).round(1).where(con_motor)
    return df


def parse_transmission_epa(df):
    """Convierte el texto ('Automatic (S8)', 'Manual 5-spd'...) en tipo y número de marchas."""
    df = df.copy()
    t = df['transmission']
    tipo = np.select([t.str.startswith('Manual'), t.str.contains('variable|AV', regex=True)],
                     ['Manual', 'CVT'], default='Automatic')
    df['transmission_type'] = pd.Series(tipo, index=df.index).where(t.notna())
    df['gears'] = pd.to_numeric(t.str.extract(r'(\d+)')[0], errors='coerce').astype('Int8')
    return df


def types_epa(df):
    return df.astype({
        'id': 'int32', 'year': 'int16',
        'make': 'category', 'vehicle_class': 'category', 'drive': 'category',
        'transmission': 'category', 'fuel_type_detail': 'category', 'fuel_type': 'category',
        'technology': 'category', 'transmission_type': 'category', 'vehicle_group': 'category',
        'engine_size_l': 'float32', 'cylinders': 'Int8',
        'city_mpg': 'int16', 'hwy_mpg': 'int16', 'comb_mpg': 'int16',
        'co2_g_mi_reported': 'Int16', 'co2_g_mi': 'float64', 'co2_g_km': 'float64',
        'fuel_comb_l100km': 'float32',
    })

Tipos elegidos (como en la Lección 2):
- `int8`, `int16`, `int32` según el rango de cada columna (cilindros ≤ 16, CO₂ ≤ 32 767 g/km, años ≤ 32 767).
- `Int8` / `Int16` con **mayúscula** (*nullable*) para columnas enteras con nulos: `cylinders` de la EPA, `gears` y `co2_g_mi_reported`.
- `category` para texto con pocos valores distintos (marca, clase, combustible, transmisión).

### 2.6 Pipeline de limpieza con registro de cambios

In [53]:
change_log = []

def log_step(df, table, step):
    """Anota filas, columnas, duplicados, nulos y memoria; devuelve la tabla sin cambios."""
    change_log.append({
        'tabla': table,
        'paso': step,
        'filas': len(df),
        'columnas': df.shape[1],
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

In [54]:
co2_clean = (co2_raw
    .pipe(log_step, 'co2_canada', '0 crudo')
    .pipe(rename_co2)
    .pipe(remove_duplicates)
    .pipe(log_step, 'co2_canada', '1 sin duplicados')
    .pipe(clean_text, ['make', 'model', 'vehicle_class', 'transmission', 'fuel_code'])
    .pipe(remove_duplicates)
    .pipe(log_step, 'co2_canada', '2 texto y duplicados implícitos')
    .pipe(parse_transmission_co2)
    .pipe(add_vehicle_group)
    .pipe(log_step, 'co2_canada', '3 transmisión y grupo')
    .pipe(types_co2)
    .pipe(log_step, 'co2_canada', '4 tipos'))

epa_clean = (epa_raw
    .pipe(log_step, 'epa', '0 crudo')
    .pipe(remove_duplicates, ignore=['id'])
    .pipe(log_step, 'epa', '1 sin duplicados')
    .pipe(select_epa)
    .pipe(log_step, 'epa', '2 columnas útiles')
    .pipe(fix_epa_values)
    .pipe(clean_text, ['make', 'model'])
    .pipe(parse_transmission_epa)
    .pipe(add_vehicle_group)
    .pipe(log_step, 'epa', '3 valores, texto y categorías')
    .pipe(types_epa)
    .pipe(log_step, 'epa', '4 tipos'))

change_report = pd.DataFrame(change_log)
change_report

,tabla,paso,filas,columnas,nulos,memoria_mb
0,co2_canada,0 crudo,7385,12,0,2.5
1,co2_canada,1 sin duplicados,6282,12,0,2.1
2,co2_canada,2 texto y duplicados implícitos,5988,12,0,2.0
3,co2_canada,3 transmisión y grupo,5988,15,224,2.7
4,co2_canada,4 tipos,5988,15,224,0.5
5,epa,0 crudo,50407,84,543858,82.3
6,epa,1 sin duplicados,50367,84,543410,82.3
7,epa,2 columnas útiles,50367,17,48016,27.3
8,epa,"3 valores, texto y categorías",50367,22,39304,35.0
9,epa,4 tipos,50367,22,39304,5.8


Si `change_report` tiene filas repetidas, ejecutaste la celda del pipeline dos veces: vuelve a ejecutar desde `change_log = []`.

**Interpretación del registro de cambios:**

- **co2_canada:** de 7 385 filas se pasa a 6 282 al quitar los duplicados exactos (–1 103) y a 5 988 al quitar los implícitos (–294). En total se eliminan 1 397 filas (18,9 %), todas configuraciones repetidas. Los 224 nulos que aparecen en el paso 3 **se crean a propósito**: son las cajas de variación continua (`AV`) que no tienen número de marchas. Con los tipos correctos la memoria baja de 2,5 a 0,5 MB (–80 %).
- **epa:** se quitan los 40 duplicados ocultos. Al conservar solo las 17 columnas útiles (de 84), los nulos bajan de 543 410 a 48 016: la mayoría estaban en columnas que el proyecto no usa (datos de vehículos eléctricos, motores alternos, etc.). Después bajan a 39 304 porque `technology` se rellena con `'Conventional'`. Los que quedan tienen una causa conocida: el centinela `-1` convertido en nulo real, la cilindrada de los eléctricos y las marchas de las cajas CVT. La memoria baja de 82,3 a 5,8 MB (–93 %).
- **Sobre la memoria:** las cifras dependen de cómo pandas 3 guarda el texto. Sin la librería `pyarrow` (como en este entorno), cada texto se guarda como un objeto de Python y ocupa más; con `pyarrow` las cifras son menores. Filas, columnas y nulos no cambian. Lo importante es la reducción: pasar texto repetido a `category` y los números a tipos más pequeños ahorra la mayor parte de la memoria.
- Al quedarse con 17 columnas, 858 vehículos de la EPA parecen repetidos, pero en el archivo original se diferencian por el motor (turbo, supercargador, *start-stop*, etc.). Son **versiones distintas** y se conservan.

### 2.7 Validación de la limpieza

In [55]:
def validate_co2(df):
    assert not df.duplicated().any(), 'co2: filas duplicadas'
    assert df.isna().sum().sum() == df['gears'].isna().sum(), 'co2: nulos fuera de gears'
    assert df['co2_g_km'].between(1, 1000).all(), 'co2: CO2 fuera de rango'
    assert df['transmission_type'].notna().all(), 'co2: transmisión sin tipo'
    assert set(df['fuel_code'].unique()) <= {'X', 'Z', 'D', 'E', 'N'}, 'co2: código de combustible desconocido'
    assert not df['model'].str.contains('  ', regex=False).any(), 'co2: espacios dobles'
    # El consumo combinado debe estar entre el de ciudad y el de carretera.
    # Se usa mínimo y máximo porque en los híbridos la ciudad gasta MENOS que la carretera
    bajo = df[['fuel_city_l100km', 'fuel_hwy_l100km']].min(axis=1)
    alto = df[['fuel_city_l100km', 'fuel_hwy_l100km']].max(axis=1)
    assert df['fuel_comb_l100km'].between(bajo, alto).all(), 'co2: consumo incoherente'
    return 'co2_canada: validación superada'


def validate_epa(df):
    assert df['id'].is_unique, 'epa: id repetido'
    # Se quitaron exactamente los duplicados ocultos detectados en 2.2
    assert len(df) == len(epa_raw) - epa_raw.duplicated(subset=cols_sin_id).sum(), 'epa: duplicados ocultos'
    assert (df['co2_g_mi_reported'].dropna() >= 0).all(), 'epa: quedan centinelas -1'  # 0 = eléctricos
    con_motor = ~df['technology'].isin(NO_ENGINE)
    assert (df.loc[con_motor, 'co2_g_km'] > 0).all(), 'epa: vehículo con motor sin CO2'
    assert df.loc[con_motor, 'engine_size_l'].isna().sum() <= 2, 'epa: nulos de cilindrada inesperados'
    assert not df['model'].str.contains('  ', regex=False).any(), 'epa: espacios dobles'
    assert df['vehicle_group'].notna().all(), 'epa: clase sin grupo'
    return 'epa: validación superada'


print(validate_co2(co2_clean))
print(validate_epa(epa_clean))

# Aplicar el pipeline otra vez a la tabla limpia no debe cambiar nada
assert len(remove_duplicates(co2_clean)) == len(co2_clean)
print(co2_clean.shape, epa_clean.shape)

co2_canada: validación superada
epa: validación superada
(5988, 15) (50367, 22)


In [56]:
co2_clean.head(3)

,make,model,vehicle_class,engine_size_l,cylinders,transmission,fuel_code,fuel_city_l100km,fuel_hwy_l100km,fuel_comb_l100km,fuel_comb_mpg,co2_g_km,transmission_type,gears,vehicle_group
0,ACURA,ILX,COMPACT,2.0,4,AS5,Z,9.9,6.7,8.5,33,196,Automatic,5,Car
1,ACURA,ILX,COMPACT,2.4,4,M6,Z,11.2,7.7,9.6,29,221,Manual,6,Car
2,ACURA,ILX HYBRID,COMPACT,1.5,4,AV7,Z,6.0,5.8,5.9,48,136,CVT,7,Car


In [57]:
epa_clean.head(3)

,id,year,make,model,vehicle_class,drive,engine_size_l,cylinders,transmission,fuel_type_detail,fuel_type,technology,city_mpg,hwy_mpg,comb_mpg,co2_g_mi_reported,co2_g_mi,co2_g_km,fuel_comb_l100km,transmission_type,gears,vehicle_group
0,1,1985,ALFA ROMEO,SPIDER VELOCE 2000,Two Seaters,Rear-Wheel Drive,2.0,4,Manual 5-spd,Regular,Regular gasoline,Conventional,19,25,21,<NA>,423.190476,263.0,11.2,Manual,5,Two-seater
1,10,1985,FERRARI,TESTAROSSA,Two Seaters,Rear-Wheel Drive,4.9,12,Manual 5-spd,Regular,Regular gasoline,Conventional,9,14,11,<NA>,807.909091,502.0,21.4,Manual,5,Two-seater
2,100,1985,DODGE,CHARGER,Subcompact Cars,Front-Wheel Drive,2.2,4,Manual 5-spd,Regular,Regular gasoline,Conventional,23,33,27,<NA>,329.148148,204.5,8.7,Manual,5,Car


## 3. Lección 3: filtrado, ordenamiento y agrupación

Cada pregunta se traduce a la forma *"métrica por grupo"*. El análisis principal usa `co2_clean` (Canadá), que ya trae el CO₂ en g/km; la EPA se usa para ver la tendencia en el tiempo.

### 3.1 Filtros con máscaras booleanas, `isin`, `between` y `query`

In [58]:
# Máscara booleana: motores grandes (8 cilindros o más)
motor_grande = co2_clean['cylinders'] >= 8
# isin: SUV y pickups
suv_pickup = co2_clean['vehicle_group'].isin(['SUV', 'Pickup'])
# between: motores pequeños de 1.0 a 2.0 L (ambos incluidos)
motor_pequeno = co2_clean['engine_size_l'].between(1.0, 2.0)

print('Motores de 8+ cilindros:', motor_grande.sum(), '| CO2 medio:', round(co2_clean.loc[motor_grande, 'co2_g_km'].mean(), 1))
print('Motores de 1.0 a 2.0 L:', motor_pequeno.sum(), '| CO2 medio:', round(co2_clean.loc[motor_pequeno, 'co2_g_km'].mean(), 1))
print('SUV o pickup con 8+ cilindros:', (suv_pickup & motor_grande).sum())

Motores de 8+ cilindros: 1341 | CO2 medio: 329.0
Motores de 1.0 a 2.0 L: 1998 | CO2 medio: 198.9
SUV o pickup con 8+ cilindros: 558


In [59]:
# query: pickups diésel frente a pickups de gasolina regular
diesel_pickup = co2_clean.query('vehicle_group == "Pickup" and fuel_code == "D"')
gas_pickup = co2_clean.query('vehicle_group == "Pickup" and fuel_code == "X"')
print('Pickups diésel:', len(diesel_pickup), '| CO2 medio:', round(diesel_pickup['co2_g_km'].mean(), 1))
print('Pickups gasolina regular:', len(gas_pickup), '| CO2 medio:', round(gas_pickup['co2_g_km'].mean(), 1))

Pickups diésel: 51 | CO2 medio: 272.7
Pickups gasolina regular: 409 | CO2 medio: 299.1


### 3.2 Ordenamiento: los que más y menos emiten

In [60]:
cols_ver = ['make', 'model', 'vehicle_group', 'engine_size_l', 'cylinders', 'fuel_code', 'co2_g_km']
print('Mayores emisiones')
display(co2_clean.nlargest(5, 'co2_g_km')[cols_ver])
print('Menores emisiones')
display(co2_clean.nsmallest(5, 'co2_g_km')[cols_ver])

Mayores emisiones


,make,model,vehicle_group,engine_size_l,cylinders,fuel_code,co2_g_km
4199,BUGATTI,CHIRON,Two-seater,8.0,16,Z,522
5170,LAMBORGHINI,AVENTADOR ROADSTER,Two-seater,6.5,12,Z,493
349,FORD,E350 WAGON,Van,6.8,10,X,488
5169,LAMBORGHINI,AVENTADOR COUPE,Two-seater,6.5,12,Z,487
5828,LAMBORGHINI,AVENTADOR COUPE,Two-seater,6.5,12,Z,485


Menores emisiones


,make,model,vehicle_group,engine_size_l,cylinders,fuel_code,co2_g_km
3581,HYUNDAI,IONIQ BLUE,Car,1.6,4,X,96
5771,HYUNDAI,IONIQ,Car,1.6,4,X,99
4450,HYUNDAI,IONIQ,Car,1.6,4,X,102
3580,HYUNDAI,IONIQ,Car,1.6,4,X,103
3019,TOYOTA,PRIUS,Car,1.8,4,X,104


Los que más emiten son deportivos de dos plazas con 12 a 16 cilindros (Bugatti Chiron, 522 g/km; Lamborghini Aventador) y una van Ford E350 con motor de 6,8 L. Los que menos emiten son híbridos (Hyundai Ioniq, Toyota Prius) con motores de 1,6 a 1,8 L. Ya se ve que el tamaño del motor separa los dos extremos.

### 3.3 Indicador 1: correlación de cada característica con el CO₂

*Métrica:* coeficiente de correlación de Pearson (de –1 a 1) **por** característica numérica. Cuanto más cerca de 1, más se mueven juntas la característica y las emisiones.

In [61]:
indicador_corr = (co2_clean[['engine_size_l', 'cylinders', 'fuel_comb_l100km', 'gears', 'co2_g_km']]
                  .astype('float64')
                  .corr()['co2_g_km']
                  .drop('co2_g_km')
                  .sort_values(ascending=False)
                  .round(3))
indicador_corr

fuel_comb_l100km    0.915
engine_size_l       0.854
cylinders           0.835
gears               0.151
Name: co2_g_km, dtype: float64

**Interpretación:**
- El **consumo de combustible** tiene la correlación más alta (0,915). Es lo esperado: el CO₂ sale directamente del combustible que se quema. Más que una característica de diseño es la consecuencia inmediata, y sirve para confirmar que los datos son coherentes.
- Entre las características de diseño, la **cilindrada** (0,854) y el **número de cilindros** (0,835) tienen una relación fuerte con las emisiones.
- El **número de marchas** casi no se relaciona (0,151).
- Una correlación alta indica que las dos variables se mueven juntas, no prueba por sí sola que una cause la otra.

### 3.4 Indicador 2: CO₂ medio por rango de cilindrada

*Métrica:* CO₂ medio (g/km) **por** rango de cilindrada. Se categoriza `engine_size_l` con `pd.cut`.

In [62]:
rangos = pd.cut(co2_clean['engine_size_l'],
                bins=[0, 2, 3, 4, 5, np.inf],
                labels=['≤ 2.0 L', '2.1–3.0 L', '3.1–4.0 L', '4.1–5.0 L', '> 5.0 L'])

indicador_cilindrada = (co2_clean
    .groupby(rangos, observed=True)
    .agg(configuraciones=('co2_g_km', 'size'),
         co2_medio=('co2_g_km', 'mean'),
         co2_mediana=('co2_g_km', 'median'),
         consumo_medio=('fuel_comb_l100km', 'mean'))
    .round(1))
indicador_cilindrada.index.name = 'cilindrada'

aumento = (indicador_cilindrada['co2_medio'].iloc[-1] / indicador_cilindrada['co2_medio'].iloc[0] - 1) * 100
print(f'Un motor de más de 5 L emite en promedio {aumento:.0f} % más CO2 por km que uno de 2 L o menos')
indicador_cilindrada

Un motor de más de 5 L emite en promedio 72 % más CO2 por km que uno de 2 L o menos


,configuraciones,co2_medio,co2_mediana,consumo_medio
cilindrada,,,,
≤ 2.0 L,2001,198.8,200.0,8.5
2.1–3.0 L,1426,239.5,240.0,10.3
3.1–4.0 L,1290,269.4,265.0,12.0
4.1–5.0 L,485,313.0,308.0,14.1
> 5.0 L,786,341.2,332.0,15.4


**Interpretación:** el CO₂ sube en todos los rangos, sin excepción: de 198,8 g/km en motores de 2 L o menos a 341,2 g/km en motores de más de 5 L (**+72 %**). Cada rango de cilindrada agrega entre 30 y 45 g/km en promedio. La mediana sigue el mismo patrón que la media, así que no son unos pocos vehículos extremos los que mueven el resultado.

### 3.5 Indicador 3: CO₂ medio por tipo de combustible y por transmisión

*Métrica:* CO₂ medio **por** combustible y **por** tipo de transmisión. Para que la comparación sea justa se muestra también la cilindrada media de cada grupo: si un grupo tiene motores más grandes, emitirá más por esa razón.

In [63]:
indicador_combustible = (co2_clean
    .groupby('fuel_code', observed=True)
    .agg(configuraciones=('co2_g_km', 'size'),
         co2_medio=('co2_g_km', 'mean'),
         cilindrada_media=('engine_size_l', 'mean'),
         consumo_medio=('fuel_comb_l100km', 'mean'))
    .round(1)
    .sort_values('co2_medio', ascending=False))
indicador_combustible

,configuraciones,co2_medio,cilindrada_media,consumo_medio
fuel_code,,,,
E,325,276.3,4.2,16.9
Z,2639,266.1,3.4,11.4
X,2877,236.4,2.8,10.1
D,146,235.6,2.5,8.8
N,1,213.0,3.6,12.7


In [64]:
indicador_transmision = (co2_clean
    .groupby('transmission_type', observed=True)
    .agg(configuraciones=('co2_g_km', 'size'),
         co2_medio=('co2_g_km', 'mean'),
         cilindrada_media=('engine_size_l', 'mean'))
    .round(1)
    .sort_values('co2_medio', ascending=False))
indicador_transmision

,configuraciones,co2_medio,cilindrada_media
transmission_type,,,
Automatic,4603,262.9,3.4
Manual,953,227.8,2.6
CVT,432,184.0,2.3


**Interpretación (combustible):** a primera vista el etanol E85 (`E`, 276,3 g/km) y la gasolina premium (`Z`, 266,1) emiten más que la regular (`X`, 236,4) y el diésel (`D`, 235,6). Pero la columna `cilindrada_media` muestra que esos grupos también tienen los motores más grandes (4,2 y 3,4 L, frente a 2,8 y 2,5 L). Buena parte de la diferencia viene del tamaño del motor, no solo del combustible. En la comparación justa de la sección 3.1 (mismo tipo de vehículo), las pickups diésel emiten 272,7 g/km frente a 299,1 g/km de las de gasolina regular (**–9 %**). El gas natural (`N`) tiene un solo vehículo, así que no permite sacar conclusiones.

**Interpretación (transmisión):** las CVT emiten menos (184,0 g/km) y las automáticas más (262,9 g/km), pero de nuevo la cilindrada media va en el mismo orden (2,3 / 2,6 / 3,4 L). Además, muchas CVT son de híbridos. La transmisión, por sí sola, no explica las emisiones.

### 3.6 Indicador 4: porcentaje de configuraciones de altas emisiones por grupo de vehículo

*Métrica:* % de configuraciones con CO₂ por encima del percentil 75 del dataset **por** grupo de vehículo.

In [65]:
umbral_alto = co2_clean['co2_g_km'].quantile(0.75)
print('Umbral de altas emisiones (percentil 75):', umbral_alto, 'g/km')

indicador_grupo = (co2_clean
    .assign(alta_emision=co2_clean['co2_g_km'] > umbral_alto)
    .groupby('vehicle_group', observed=True)
    .agg(configuraciones=('co2_g_km', 'size'),
         co2_medio=('co2_g_km', 'mean'),
         pct_altas_emisiones=('alta_emision', 'mean'))
    .assign(pct_altas_emisiones=lambda t: t['pct_altas_emisiones'] * 100)
    .round(1)
    .sort_values('co2_medio', ascending=False))
indicador_grupo

Umbral de altas emisiones (percentil 75): 290.0 g/km


,configuraciones,co2_medio,pct_altas_emisiones
vehicle_group,,,
Van,88,388.3,100.0
Pickup,588,296.2,51.9
Two-seater,352,284.1,44.9
SUV,1550,263.2,28.7
Minivan,54,261.1,3.7
Special purpose,61,237.5,3.3
Car,3051,232.8,15.0
Station wagon,244,210.8,4.5


**Interpretación:** el umbral de altas emisiones es 290 g/km. El **100 %** de las vans de este dataset lo supera, al igual que la mitad de las pickups (51,9 %) y casi la mitad de los deportivos de dos plazas (44,9 %). En cambio, solo el 15,0 % de los automóviles y el 4,5 % de los station wagon lo superan. El tipo de carrocería también influye en las emisiones.

### 3.7 Tabla dinámica: grupo de vehículo × cilindros

¿El tamaño del motor explica las emisiones *dentro* de cada tipo de vehículo, o solo es que los SUV y pickups tienen motores grandes?

In [66]:
matriz = co2_clean.pivot_table(index='vehicle_group', columns='cylinders',
                               values='co2_g_km', aggfunc='mean', observed=True).round(0)
print(matriz.shape)
matriz[[4, 6, 8]]

(8, 8)


cylinders,4,6,8
vehicle_group,,,
Car,194.0,250.0,305.0
Minivan,226.0,266.0,NaN
Pickup,266.0,283.0,320.0
SUV,222.0,276.0,331.0
Special purpose,234.0,289.0,NaN
Station wagon,198.0,251.0,321.0
Two-seater,202.0,267.0,317.0
Van,NaN,342.0,400.0


**Interpretación:** dentro de **cada** grupo, el CO₂ sube con los cilindros. En automóviles pasa de 194 a 250 y a 305 g/km (4, 6 y 8 cilindros); en SUV, de 222 a 276 y a 331 g/km. O sea, el tamaño del motor influye aunque se compare el mismo tipo de vehículo. Y con igual número de cilindros, las carrocerías más grandes emiten más (con 8 cilindros, una van llega a 400 g/km y un automóvil a 305). Las dos características suman su efecto.

Las celdas vacías (`NaN`) son combinaciones que no existen en los datos, por ejemplo minivans de 8 cilindros.

### 3.8 Tendencia en el tiempo con la EPA

El dataset de Canadá no tiene año, así que la evolución se mira en la EPA. Solo vehículos con motor de combustión (`~isin(NO_ENGINE)`).

In [67]:
combustion = epa_clean[~epa_clean['technology'].isin(NO_ENGINE)]

tendencia = (combustion
    .groupby('year')
    .agg(modelos=('co2_g_km', 'size'), co2_medio=('co2_g_km', 'mean'),
         cilindrada_media=('engine_size_l', 'mean'))
    .round(1))
tendencia.loc[[1985, 1995, 2005, 2014, 2020, 2025]]

,modelos,co2_medio,cilindrada_media
year,,,
1985,1700,302.7,3.2
1995,961,311.9,3.5
2005,1166,302.0,3.4
2014,1206,260.6,3.3
2020,1216,248.2,3.1
2025,943,240.1,3.0


In [68]:
# Mismo periodo que el dataset de Canadá
epa_2014_2020 = combustion.query('2014 <= year <= 2020')
print('Modelos EPA 2014-2020:', len(epa_2014_2020),
      '| CO2 medio:', round(epa_2014_2020['co2_g_km'].mean(), 1), 'g/km',
      '| Canadá:', round(co2_clean['co2_g_km'].mean(), 1), 'g/km')

Modelos EPA 2014-2020: 8810 | CO2 medio: 251.1 g/km | Canadá: 251.6 g/km


**Interpretación:** en la EPA, los vehículos de combustión emitían en promedio alrededor de 300–312 g/km entre 1985 y 2005, y la cifra baja a 260,6 (2014), 248,2 (2020) y 240,1 g/km (2025). En el mismo periodo la cilindrada media pasó de 3,4–3,5 L a 3,0 L, lo que coincide con el hallazgo principal.

Para 2014–2020 el promedio de EE. UU. (251,1 g/km) es casi igual al de Canadá (251,6 g/km). Esto respalda que los dos datasets son comparables y se pueden combinar en la Lección 4.

### 3.9 Conclusión del análisis

Respuesta a la pregunta analítica con los indicadores:

| Indicador | Resultado | Qué dice |
| --- | --- | --- |
| 1. Correlación con el CO₂ | Cilindrada 0,854 · cilindros 0,835 · marchas 0,151 | El tamaño del motor es la característica de diseño más relacionada |
| 2. CO₂ por rango de cilindrada | 198,8 → 341,2 g/km (+72 %) | Más litros, más emisiones, sin excepción |
| 3. CO₂ por combustible y transmisión | Diésel –9 % en pickups; diferencias de transmisión explicadas por la cilindrada | Influyen menos que el motor |
| 4. % de altas emisiones por grupo | Vans 100 %, pickups 51,9 %, automóviles 15,0 % | La carrocería también influye |

**Conclusión:** las características que más influyen en las emisiones de CO₂ por km son la **cilindrada** y el **número de cilindros**, seguidas del **tipo de vehículo** (vans, pickups y deportivos). El combustible y la transmisión muestran diferencias, pero en buena parte se explican porque cada grupo tiene motores de distinto tamaño. El consumo de combustible es la variable más ligada al CO₂ porque es su causa directa.